# Test retrieve result

Notebook n?y ch? d?ng ?? xem k?t qu? `rag.retrieve` c? ??ng kh?ng. Kh?ng g?i answer agent.


In [1]:
from __future__ import annotations

import html
import sys
from pathlib import Path
from IPython.display import HTML, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "pipeline").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipeline.rag.legal_rag.config import RetrieveConfig
from src.pipeline.rag.legal_rag.qdrant_settings import apply_qdrant_settings, qdrant_target_label
from src.pipeline.rag.legal_rag.retriever import ChunkRetriever


D:\Users\ADMIN\miniconda3\envs\py3.10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
query = "xe mô tô "
mode = "hybrid"  # dense | sparse | hybrid
top_k = 10
as_of = None  # vi du: "2025-01-01"
document_number = None  # vi du: "168/2024/ND-CP"
use_local_qdrant = False


In [3]:
config = RetrieveConfig(
    qdrant_path=ROOT / "data" / "preprocessed" / "qdrant",
    collection_name="legal_chunks",
    mode=mode,
    top_k=top_k,
)
apply_qdrant_settings(config, use_local=use_local_qdrant)

retriever = ChunkRetriever(config)
results = retriever.retrieve(
    query,
    as_of_date=as_of,
    document_number=document_number,
    mode=mode,
    top_k=top_k,
)

print("target:", qdrant_target_label(config))
print("results:", len(results))


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]D:\Users\ADMIN\miniconda3\envs\py3.10\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\AppData\Local\Temp\fastembed_cache\models--Qdrant--bm25. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 18 files: 100%|██████████| 18/18 [00:01<0

target: cloud:https://d406ab3d-a4f4-429a-bb20-b765333f6d6b.sa-east-1-0.aws.cloud.qdrant.io/
results: 10


In [4]:
def render_result(row: dict, idx: int) -> str:
    content = html.escape(row.get("content") or "")
    meta = html.escape(
        f"score={row.get('score'):.4f} | mode={row.get('mode')} | "
        f"doc={row.get('document_number')} | effective={row.get('effective_from')} -> {row.get('effective_to')}"
    )
    path = html.escape(row.get("path_text") or "")
    chunk_id = html.escape(str(row.get("chunk_id") or ""))
    return "\n".join([
        '<section style="border:1px solid #ddd; border-radius:8px; padding:12px; margin:12px 0;">',
        f'<h3 style="margin:0 0 8px 0;">#{idx}</h3>',
        f'<div><b>{meta}</b></div>',
        f'<div><b>path:</b> {path}</div>',
        f'<div><b>chunk_id:</b> {chunk_id}</div>',
        f'<pre style="white-space:pre-wrap; overflow:visible; margin-top:10px;">{content}</pre>',
        '</section>',
    ])

html_output = "".join(render_result(row, i) for i, row in enumerate(results, 1))
display(HTML(html_output or "<b>No results</b>"))
